In [1]:
import os
from dotenv import load_dotenv

# 현재 폴더에 있는 .env 파일을 읽어서 환경 변수로 등록
load_dotenv()

True

In [2]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console

In [3]:
model = OpenAIChatCompletionClient(model='gpt-4o-mini')

# Crewai랑 다른 것은 backstroy, goal을 적을 필요가 없음
# model이 먼지만 정의해주면 됨. brain만 정의해주면 됨
clarity_agent = AssistantAgent(
    "ClarityAgent",
    model_client=model,
    system_message="""You are an expert editor focused on clarity and simplicity. 
            Your job is to eliminate ambiguity, redundancy, and make every sentence crisp and clear. 
            Don't worry about persuasion or tone — just make the message easy to read and understand.""",
)

# 어조, 이메일의 어조 개선
tone_agent = AssistantAgent(
    "ToneAgent",
    model_client=model,
    system_message="""You are a communication coach focused on emotional tone and professionalism. 
            Your job is to make the email sound warm, confident, and human — while staying professional 
            and appropriate for the audience. Improve the emotional resonance, polish the phrasing, 
            and adjust any words that may come off as stiff, cold, or overly casual.""",
)

# 설득, 이메일을 더 설득력 있게 개선
persuasion_agent = AssistantAgent(
    "PersuasionAgent",
    model_client=model,
    system_message="""You are a persuasion expert trained in marketing, behavioral psychology, 
            and copywriting. Your job is to enhance the email's persuasive power: improve call to action, structure arguments, and emphasize benefits. Remove weak or passive language.""",
)

# 합성, 위 모든 아이디어를 종합해서 이메일 작성
synthesizer_agent = AssistantAgent(
    "SynthesizerAgent",
    model_client=model,
    system_message="""You are an advanced email-writing specialist. Your role is to read all 
            prior agent responses and revisions, and then **synthesize the best ideas** into a unified, 
            polished draft of the email. Focus on: Integrating clarity, tone, and persuasion improvements; 
            Ensuring coherence, fluency, and a natural voice; Creating a version that feels professional, 
            effective, and readable.""",
)

# 최종 결과, 여기서 평가해서 별로면 다시 작성
critic_agent = AssistantAgent(
    "CriticAgent",
    model_client=model,
    system_message="""You are an email quality evaluator. Your job is to perform a final review 
            of the synthesized email and determine if it meets professional standards. Review the email for: 
            Clarity and flow, appropriate professional tone, effective call-to-action, and overall coherence.
            Be constructive but decisive. If the email has major flaws (unclear message, unprofessional tone, 
            or missing key elements), provide ONE specific improvement suggestion. If the email meets professional standards and communicates effectively, respond with 'The email meets professional standards.' followed by `TERMINATE` on a new line. You should only approve emails that are perfect enough for professional use, dont settle.
            
            Final result(All contents) must be written in Korean
            """,
)

In [4]:
text_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=30)

termination_condition = text_termination | max_messages_termination

In [5]:
team = RoundRobinGroupChat(
    participants = [
        clarity_agent,
        tone_agent,
        persuasion_agent,
        synthesizer_agent,
        critic_agent,
    ],
    termination_condition = termination_condition,
)

# asynchronus Python 사용하기 때문에 await써야함
# 그냥 run 사용하면 결과 다 나오고 메세지가 나오지만
# run_stream은 결과 나오기 전에, 메세지나오는대로 다 보여줌
await Console(
    team.run_stream(
        task = '안녕하세요. [귀하]의 회사 AI 분석 후 결과 해석 능력이 마음에 들어 도움을 청하고자 메일을 드립니다.'
    )
)

---------- TextMessage (user) ----------
안녕하세요. [귀하]의 회사 AI 분석 후 결과 해석 능력이 마음에 들어 도움을 청하고자 메일을 드립니다.
---------- TextMessage (ClarityAgent) ----------
안녕하세요. [귀하]의 회사 AI 분석 결과 해석 능력이 뛰어나 도움을 요청하고자 합니다.
---------- TextMessage (ToneAgent) ----------
안녕하세요. 

[귀하]의 회사가 제공하는 AI 분석 결과 해석 능력에 깊은 인상을 받았습니다. 이에 도움을 요청드리고자 이렇게 메일을 드립니다. 

감사합니다.
---------- TextMessage (PersuasionAgent) ----------
안녕하세요,

[귀하]의 회사가 제공하는 AI 분석 결과 해석 능력에 깊은 감명을 받았습니다. 여러분의 전문성을 통해 저희도 큰 이익을 얻을 수 있을 것이라고 확신합니다.

이번 기회를 통해 특정 과제에 대해 귀하의 통찰력 있는 분석을 받고자 합니다. 이를 통해 더욱 효과적인 전략을 수립하고자 합니다. 궁금한 점이나 논의할 사항이 있다면 말씀해 주시면 감사하겠습니다. 

함께 협력하여 시너지 효과를 만들어나갈 수 있기를 기대합니다. 답변 주시면 감사하겠습니다.

감사합니다. 

[귀하의 이름]  
[귀하의 회사]  
[연락처 정보]
---------- TextMessage (SynthesizerAgent) ----------
안녕하세요,

[귀하]의 회사가 제공하는 AI 분석 결과 해석 능력에 깊은 감명을 받았습니다. 여러분의 전문성을 통해 저희도 큰 이익을 얻을 수 있을 것이라고 확신합니다.

이번 기회를 통해 특정 과제에 대해 귀하의 통찰력 있는 분석을 받고자 합니다. 이를 통해 더욱 효과적인 전략을 수립하고, 시너지 효과를 만들어 나가길 기대합니다.

궁금한 점이나 논의하고 싶은 사항이 있으시다면 언제든지 말씀해 주시면 감사하겠습니다. 답변 기다리겠습니다.

감사합니다.

TaskResult(messages=[TextMessage(id='e67e033f-0c4f-42a5-a468-386a6ad26055', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 3, 1, 13, 53, 5, 744867, tzinfo=datetime.timezone.utc), content='안녕하세요. [귀하]의 회사 AI 분석 후 결과 해석 능력이 마음에 들어 도움을 청하고자 메일을 드립니다.', type='TextMessage'), TextMessage(id='73da643c-c6cd-41e8-b4e0-9b2077397a39', source='ClarityAgent', models_usage=RequestUsage(prompt_tokens=92, completion_tokens=25), metadata={}, created_at=datetime.datetime(2026, 3, 1, 13, 53, 7, 853777, tzinfo=datetime.timezone.utc), content='안녕하세요. [귀하]의 회사 AI 분석 결과 해석 능력이 뛰어나 도움을 요청하고자 합니다.', type='TextMessage'), TextMessage(id='6087cec2-bd9f-452d-9b8d-bfbfe4f645c9', source='ToneAgent', models_usage=RequestUsage(prompt_tokens=145, completion_tokens=47), metadata={}, created_at=datetime.datetime(2026, 3, 1, 13, 53, 9, 395348, tzinfo=datetime.timezone.utc), content='안녕하세요. \n\n[귀하]의 회사가 제공하는 AI 분석 결과 해석 능력에 깊은 인상을 받았습니다. 이에 도움을 요청드리고자 이렇게 메일을 드립니다. \n\n감사합니다.', type='Tex